# 3-Fold CV: Supervised CNN Baseline (EfficientNet-B3, ImageNet Pretrained)

A second supervised CNN baseline, complementary to ResNet-50, for additional context in the foundation model comparison.

**Purpose:** demonstrates the typical performance range achievable from standard supervised CNN approaches with ImageNet pretraining. Two architectures (ResNet-50, EfficientNet-B3) trained with the same protocol provide a more robust anchor than a single CNN baseline.

**Model:** EfficientNet-B3 with ImageNet-1K weights (~12M parameters, smaller than ResNet-50's ~25M but more efficient per parameter). Classification head: linear layer mapping 1536-dim pooled features to 14 classes.

**Training:** all 14 classes, end-to-end fine-tuning with the same hyperparameter strategy as ResNet-50 for fair comparison.

**Outputs:**
- `output_efficientnetb3/per_fold_per_class_results.csv`
- `output_efficientnetb3/per_class_summary_mean_std.csv`
- `output_efficientnetb3/headline_metrics_mean_std.csv`

## Imports and configuration

In [1]:
import re
import time
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import classification_report
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch: 2.8.0+cu128
CUDA: True
GPU: NVIDIA A100-SXM4-80GB


In [2]:
CONFIG = {
    "device": "cuda" if torch.cuda.is_available() else "cpu",

    "meta_path": "/workspace/thesis/ultimate_bioclip_top1_rank_order_meta_12062025.csv",
    "image_root": "/workspace/thesis/ALL_copy",
    "output_dir": "/workspace/thesis/output_efficientnetb3",

    "image_id_col": "image_id",
    "label_col": "label",

    "k_folds": 3,
    "random_state": 0,
    "min_class_size": 50,

    # Training hyperparameters (same as ResNet-50 for fair comparison)
    "epochs": 5,
    "batch_size": 32,
    "lr_backbone": 1e-4,
    "lr_head": 1e-3,
    "weight_decay": 1e-4,
    "num_workers": 4,

    # EfficientNet-B3 expects 300x300 input (vs ResNet-50's 224x224)
    # This is the native training resolution for B3
    "image_size": 300,
    "imagenet_mean": [0.485, 0.456, 0.406],
    "imagenet_std": [0.229, 0.224, 0.225],
}

CROP_ID_PATTERN = re.compile(r"(\d{14}_crop_[a-zA-Z0-9]+\.jpg)$")
Path(CONFIG["output_dir"]).mkdir(parents=True, exist_ok=True)

print(f"Hyperparameters: epochs={CONFIG['epochs']}, batch_size={CONFIG['batch_size']}")
print(f"Learning rates: backbone={CONFIG['lr_backbone']}, head={CONFIG['lr_head']}")
print(f"Image size: {CONFIG['image_size']} (EfficientNet-B3 native)")
print(f"Output: {CONFIG['output_dir']}")

Hyperparameters: epochs=5, batch_size=32
Learning rates: backbone=0.0001, head=0.001
Image size: 300 (EfficientNet-B3 native)
Output: /workspace/thesis/output_efficientnetb3


## Preprocessing pipelines

EfficientNet-B3 was pretrained at 300×300, larger than ResNet-50's 224×224. Using the native resolution typically gives best transfer learning performance.

Two pipelines:
- **Training:** with augmentation (random crops, horizontal flip, color jitter)
- **Eval:** deterministic resize + center crop

In [3]:
train_transform = transforms.Compose([
    transforms.Resize((CONFIG["image_size"] + 40, CONFIG["image_size"] + 40),
                      interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.RandomCrop(CONFIG["image_size"]),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=CONFIG["imagenet_mean"], std=CONFIG["imagenet_std"]),
])

eval_transform = transforms.Compose([
    transforms.Resize(CONFIG["image_size"], interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(CONFIG["image_size"]),
    transforms.ToTensor(),
    transforms.Normalize(mean=CONFIG["imagenet_mean"], std=CONFIG["imagenet_std"]),
])

print("Train transform:")
print(train_transform)
print("\nEval transform:")
print(eval_transform)

Train transform:
Compose(
    Resize(size=(340, 340), interpolation=bicubic, max_size=None, antialias=True)
    RandomCrop(size=(300, 300), padding=None)
    RandomHorizontalFlip(p=0.5)
    ColorJitter(brightness=(0.8, 1.2), contrast=(0.8, 1.2), saturation=(0.8, 1.2), hue=None)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)

Eval transform:
Compose(
    Resize(size=300, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(300, 300))
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


## Helper functions

In [4]:
def build_path_index(image_root):
    """Scan local image directory, build {image_id: full_path}."""
    image_root = Path(image_root)
    files = list(image_root.glob("*.jpg"))
    path_index = {}
    for f in files:
        m = CROP_ID_PATTERN.search(f.name)
        if m:
            path_index[m.group(1)] = str(f)
    print(f"Mapped {len(path_index)} image_ids")
    return path_index


class ImageOrderDataset(Dataset):
    """Dataset of (image, label) pairs. Transform passed at construction."""
    def __init__(self, image_paths, labels, transform):
        self.image_paths = image_paths
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        img = self.transform(img)
        return img, self.labels[idx]

In [5]:
def build_efficientnet_b3(n_classes, device):
    """Build EfficientNet-B3 with ImageNet weights and new classification head."""
    weights = models.EfficientNet_B3_Weights.IMAGENET1K_V1
    model = models.efficientnet_b3(weights=weights)

    # EfficientNet's classifier is a Sequential: [Dropout, Linear].
    # We replace the final Linear (1000 ImageNet classes -> n_classes), preserving dropout.
    in_features = model.classifier[-1].in_features  # 1536 for EfficientNet-B3
    model.classifier[-1] = nn.Linear(in_features, n_classes)

    return model.to(device)


def get_param_groups(model, config):
    """Separate backbone and head parameters for different learning rates.

    EfficientNet's structure: features (backbone) + avgpool + classifier (head).
    The 'classifier' attribute contains the new Linear layer (and dropout).
    """
    backbone_params = []
    head_params = []
    for name, param in model.named_parameters():
        if name.startswith("classifier."):
            head_params.append(param)
        else:
            backbone_params.append(param)

    return [
        {"params": backbone_params, "lr": config["lr_backbone"]},
        {"params": head_params, "lr": config["lr_head"]},
    ]


def train_efficientnet(model, train_loader, config, fold_idx):
    """Fine-tune EfficientNet-B3 with cross-entropy loss."""
    device = config["device"]

    param_groups = get_param_groups(model, config)
    optimizer = torch.optim.AdamW(param_groups, weight_decay=config["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["epochs"])

    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Trainable parameters: {n_trainable:,}")

    for epoch in range(config["epochs"]):
        model.train()
        total_loss, n_batches, n_correct, n_total = 0.0, 0, 0, 0
        ep_start = time.time()

        for batch_idx, (imgs, labels) in enumerate(train_loader):
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(imgs)
            loss = F.cross_entropy(logits, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            n_batches += 1
            n_correct += (logits.argmax(dim=-1) == labels).sum().item()
            n_total += labels.size(0)

            if batch_idx % 100 == 0:
                acc = n_correct / max(n_total, 1)
                print(f"    Fold {fold_idx+1} Ep {epoch+1} batch {batch_idx}/{len(train_loader)}: "
                      f"loss={loss.item():.4f}, running_acc={acc:.4f}")

        scheduler.step()
        avg_loss = total_loss / n_batches
        acc = n_correct / n_total
        print(f"    Fold {fold_idx+1} Ep {epoch+1}: avg_loss={avg_loss:.4f}, train_acc={acc:.4f} "
              f"({(time.time()-ep_start)/60:.1f} min)")

    return model


def evaluate_efficientnet(model, val_dataset, config):
    """Run inference on val set, return predictions."""
    device = config["device"]
    model.eval()

    loader = DataLoader(
        val_dataset, batch_size=config["batch_size"], shuffle=False,
        num_workers=config["num_workers"], pin_memory=True,
    )

    all_preds = []
    with torch.no_grad():
        for imgs, _ in loader:
            imgs = imgs.to(device, non_blocking=True)
            logits = model(imgs)
            preds = logits.argmax(dim=-1).cpu().numpy()
            all_preds.append(preds)

    return np.concatenate(all_preds, axis=0)

## Load metadata

In [6]:
path_index = build_path_index(CONFIG["image_root"])

meta = pd.read_csv(CONFIG["meta_path"])
meta = meta.dropna(subset=[CONFIG["label_col"], CONFIG["image_id_col"]]).reset_index(drop=True)
meta["local_path"] = meta[CONFIG["image_id_col"]].map(path_index)
matched = meta.dropna(subset=["local_path"]).reset_index(drop=True)

y_str = matched[CONFIG["label_col"]].to_numpy()
image_paths = matched["local_path"].to_numpy()
classnames = sorted(np.unique(y_str).tolist())
class_to_idx = {c: i for i, c in enumerate(classnames)}
y = np.array([class_to_idx[s] for s in y_str])

class_counts = pd.Series(y_str).value_counts()
eligible_classes = class_counts[class_counts >= CONFIG["min_class_size"]].index.tolist()
eligible_class_set = set(eligible_classes)

print(f"Matched: {len(matched)} crops")
print(f"Classes ({len(classnames)}): {classnames}")
print(f"Eligible (n>=50, for macro_f1_eligible metric): {eligible_classes}")

Mapped 48095 image_ids
Matched: 39787 crops
Classes (14): ['Araneae', 'Blattodea', 'Coleoptera', 'Diptera', 'Hemiptera', 'Hymenoptera', 'Ixodida', 'Lepidoptera', 'Mecoptera', 'Neuroptera', 'Orthoptera', 'Plecoptera', 'Raphidioptera', 'Trichoptera']
Eligible (n>=50, for macro_f1_eligible metric): ['Diptera', 'Hymenoptera', 'Lepidoptera', 'Coleoptera', 'Hemiptera', 'Araneae', 'Neuroptera', 'Trichoptera', 'Plecoptera']


## Set up 3-fold stratified CV

Same protocol as other notebooks for direct comparability. Classes with n<3 (Ixodida n=1, Raphidioptera n=2) kept in training every fold.

In [7]:
class_counts_arr = np.bincount(y, minlength=len(classnames))
splittable_mask = class_counts_arr[y] >= CONFIG["k_folds"]
splittable_indices = np.where(splittable_mask)[0]
unsplittable_indices = np.where(~splittable_mask)[0]

print(f"Splittable samples (n>={CONFIG['k_folds']}): {len(splittable_indices)}")
print(f"Unsplittable samples (kept in train): {len(unsplittable_indices)}")

skf = StratifiedKFold(n_splits=CONFIG["k_folds"], shuffle=True, random_state=CONFIG["random_state"])
raw_splits = list(skf.split(splittable_indices, y[splittable_indices]))

fold_splits = []
for train_rel, val_rel in raw_splits:
    train_idx = np.concatenate([splittable_indices[train_rel], unsplittable_indices])
    val_idx = splittable_indices[val_rel]
    fold_splits.append((train_idx, val_idx))

for i, (tr, va) in enumerate(fold_splits):
    print(f"  Fold {i+1}: train={len(tr)}, val={len(va)}")

Splittable samples (n>=3): 39784
Unsplittable samples (kept in train): 3
  Fold 1: train=26525, val=13262
  Fold 2: train=26526, val=13261
  Fold 3: train=26526, val=13261


## Run cross-validation

For each fold: train EfficientNet-B3 fresh on the fold's training set, evaluate on val set.

EfficientNet-B3 trains on **all 14 classes** with cross-entropy loss. Same protocol as ResNet-50 for fair comparison.

**Note on memory:** EfficientNet-B3 at 300×300 input uses more GPU memory than ResNet-50 at 224×224. If you encounter OOM, reduce `batch_size` to 24 or 16 in CONFIG.

In [8]:
all_results = []
overall_start = time.time()

for fold_idx, (train_idx, val_idx) in enumerate(fold_splits):
    print(f"\n{'='*70}\nFOLD {fold_idx+1}/{CONFIG['k_folds']}\n{'='*70}")
    fold_start = time.time()

    train_paths = image_paths[train_idx]
    train_labels = y[train_idx]
    val_paths = image_paths[val_idx]
    val_labels = y[val_idx]

    print(f"  Train: {len(train_idx)}, Val: {len(val_idx)}")

    train_dataset = ImageOrderDataset(train_paths, train_labels, train_transform)
    val_dataset = ImageOrderDataset(val_paths, val_labels, eval_transform)

    train_loader = DataLoader(
        train_dataset, batch_size=CONFIG["batch_size"],
        shuffle=True, num_workers=CONFIG["num_workers"], pin_memory=True,
    )

    # Fresh model per fold (ImageNet pretrained weights)
    print(f"\n  Building fresh EfficientNet-B3 with ImageNet weights...")
    model = build_efficientnet_b3(len(classnames), CONFIG["device"])

    # Train
    print(f"\n  Training EfficientNet-B3 for {CONFIG['epochs']} epochs...")
    train_start = time.time()
    model = train_efficientnet(model, train_loader, CONFIG, fold_idx)
    print(f"  Training time: {timedelta(seconds=int(time.time() - train_start))}")

    # Evaluate
    print(f"\n  Evaluating on val set...")
    eval_start = time.time()
    preds = evaluate_efficientnet(model, val_dataset, CONFIG)
    print(f"  Eval time: {timedelta(seconds=int(time.time() - eval_start))}")

    # Classification report
    report = classification_report(
        val_labels, preds, labels=list(range(len(classnames))),
        target_names=classnames, output_dict=True, zero_division=0,
    )

    for cls_name in classnames:
        stats = report.get(cls_name, {})
        all_results.append({
            "method": "efficientnetb3_finetuned",
            "fold": fold_idx,
            "class": cls_name,
            "precision": round(stats.get("precision", 0.0), 4),
            "recall": round(stats.get("recall", 0.0), 4),
            "f1": round(stats.get("f1-score", 0.0), 4),
            "support": int(stats.get("support", 0)),
        })

    # Save intermediate
    pd.DataFrame(all_results).to_csv(
        Path(CONFIG["output_dir"]) / "per_fold_per_class_results.csv", index=False
    )

    # Cleanup
    del model
    torch.cuda.empty_cache()

    print(f"\n  Fold {fold_idx+1} total: {timedelta(seconds=int(time.time() - fold_start))}")

print(f"\n{'='*70}\nCV COMPLETE — Total: {timedelta(seconds=int(time.time() - overall_start))}\n{'='*70}")


FOLD 1/3
  Train: 26525, Val: 13262

  Building fresh EfficientNet-B3 with ImageNet weights...
Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 112MB/s] 



  Training EfficientNet-B3 for 5 epochs...
  Trainable parameters: 10,717,750
    Fold 1 Ep 1 batch 0/829: loss=2.6260, running_acc=0.0625
    Fold 1 Ep 1 batch 100/829: loss=0.8604, running_acc=0.7995
    Fold 1 Ep 1 batch 200/829: loss=0.4444, running_acc=0.8397
    Fold 1 Ep 1 batch 300/829: loss=0.2268, running_acc=0.8624
    Fold 1 Ep 1 batch 400/829: loss=0.2080, running_acc=0.8752
    Fold 1 Ep 1 batch 500/829: loss=0.0916, running_acc=0.8841
    Fold 1 Ep 1 batch 600/829: loss=0.0660, running_acc=0.8929
    Fold 1 Ep 1 batch 700/829: loss=0.0760, running_acc=0.8999
    Fold 1 Ep 1 batch 800/829: loss=0.1405, running_acc=0.9052
    Fold 1 Ep 1: avg_loss=0.3401, train_acc=0.9062 (3.2 min)
    Fold 1 Ep 2 batch 0/829: loss=0.2178, running_acc=0.9375
    Fold 1 Ep 2 batch 100/829: loss=0.1209, running_acc=0.9545
    Fold 1 Ep 2 batch 200/829: loss=0.0582, running_acc=0.9549
    Fold 1 Ep 2 batch 300/829: loss=0.2847, running_acc=0.9555
    Fold 1 Ep 2 batch 400/829: loss=0.1862, r

## Aggregate: per-class mean and std

In [9]:
results_df = pd.DataFrame(all_results)

summary_rows = []
for cls_name in classnames:
    class_data = results_df[results_df["class"] == cls_name]
    summary_rows.append({
        "class": cls_name,
        "support_total": int(class_data["support"].sum()),
        "f1_mean": round(class_data["f1"].mean(), 4),
        "f1_std": round(class_data["f1"].std(ddof=1), 4),
        "precision_mean": round(class_data["precision"].mean(), 4),
        "precision_std": round(class_data["precision"].std(ddof=1), 4),
        "recall_mean": round(class_data["recall"].mean(), 4),
        "recall_std": round(class_data["recall"].std(ddof=1), 4),
    })
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(Path(CONFIG["output_dir"]) / "per_class_summary_mean_std.csv", index=False)
summary_df

,class,support_total,f1_mean,f1_std,precision_mean,precision_std,recall_mean,recall_std
0,Araneae,564,0.9784,0.0100,0.9911,0.0082,0.9663,0.0240
1,Blattodea,23,0.8855,0.0742,0.9524,0.0825,0.8274,0.0676
2,Coleoptera,1143,0.8558,0.0121,0.8606,0.0199,0.8512,0.0076
3,Diptera,31913,0.9864,0.0001,0.9866,0.0010,0.9863,0.0009
4,Hemiptera,775,0.8564,0.0180,0.8952,0.0124,0.8220,0.0438
5,Hymenoptera,3326,0.9275,0.0031,0.9171,0.0046,0.9381,0.0079
6,Ixodida,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
7,Lepidoptera,1744,0.9754,0.0035,0.9716,0.0021,0.9794,0.0052
8,Mecoptera,34,0.4667,0.1014,0.5726,0.1829,0.4368,0.1561
9,Neuroptera,115,0.6634,0.0277,0.6643,0.0568,0.6701,0.0811


## Headline aggregate metrics

In [10]:
per_fold_weighted = []
per_fold_macro_all = []
per_fold_macro_eligible = []
per_fold_hemi = []
per_fold_cole = []

for fold in range(CONFIG["k_folds"]):
    fold_data = results_df[results_df["fold"] == fold]
    per_fold_weighted.append(np.average(fold_data["f1"], weights=fold_data["support"]))
    per_fold_macro_all.append(fold_data["f1"].mean())
    eligible_data = fold_data[fold_data["class"].isin(eligible_class_set)]
    per_fold_macro_eligible.append(eligible_data["f1"].mean())
    per_fold_hemi.append(fold_data[fold_data["class"] == "Hemiptera"]["f1"].iloc[0])
    per_fold_cole.append(fold_data[fold_data["class"] == "Coleoptera"]["f1"].iloc[0])

headline_df = pd.DataFrame([{
    "method": "efficientnetb3_finetuned",
    "weighted_f1_mean": round(np.mean(per_fold_weighted), 4),
    "weighted_f1_std": round(np.std(per_fold_weighted, ddof=1), 4),
    "macro_f1_all_mean": round(np.mean(per_fold_macro_all), 4),
    "macro_f1_all_std": round(np.std(per_fold_macro_all, ddof=1), 4),
    "macro_f1_eligible_mean": round(np.mean(per_fold_macro_eligible), 4),
    "macro_f1_eligible_std": round(np.std(per_fold_macro_eligible, ddof=1), 4),
    "Hemiptera_f1_mean": round(np.mean(per_fold_hemi), 4),
    "Hemiptera_f1_std": round(np.std(per_fold_hemi, ddof=1), 4),
    "Coleoptera_f1_mean": round(np.mean(per_fold_cole), 4),
    "Coleoptera_f1_std": round(np.std(per_fold_cole, ddof=1), 4),
}])
headline_df.to_csv(Path(CONFIG["output_dir"]) / "headline_metrics_mean_std.csv", index=False)
headline_df

,method,weighted_f1_mean,weighted_f1_std,macro_f1_all_mean,macro_f1_all_std,macro_f1_eligible_mean,macro_f1_eligible_std,Hemiptera_f1_mean,Hemiptera_f1_std,Coleoptera_f1_mean,Coleoptera_f1_std
0,efficientnetb3_finetuned,0.9727,0.0005,0.7318,0.0076,0.8845,0.0019,0.8564,0.018,0.8558,0.0121


## Output files summary

EfficientNet-B3 (ImageNet pretrained, fine-tuned end-to-end at 300×300) results saved to `output_efficientnetb3/`.

The notebook generates:

| File | Purpose |
|---|---|
| `per_fold_per_class_results.csv` | Granular per-fold per-class precision/recall/F1 |
| `per_class_summary_mean_std.csv` | Per-class aggregates with mean ± std across folds |
| `headline_metrics_mean_std.csv` | Top-line aggregates (weighted F1, macro F1, etc.) with mean ± std |

Second supervised CNN baseline (alongside ResNet-50). Two CNN backbones with the same training protocol provide robustness against backbone-specific artifacts in the supervised-baseline anchor.

Cross-architecture comparison: if ResNet-50 and EfficientNet-B3 produce similar headline metrics, the supervised CNN baseline characterization is robust. If they differ substantially, that itself is an interesting finding about how backbone architecture interacts with the task.
